In [2]:
import geemap
import utils as F
import ee
import concurrent.futures
import glob
import rasterio
import geopandas as gpd
from rasterio.merge import merge
from shapely.geometry import mapping
from shapely.ops import unary_union, transform
import shapely
import concurrent.futures
import os

import ee
os.environ["http_proxy"] = "http://127.0.0.1:7890"
os.environ["https_proxy"] = "http://127.0.0.1:7890"
ee.Initialize(project="ee-103dg017")
ee.Authenticate()

True

In [3]:
def force_2d(geom):
    # 兼容 Shapely 1.x/2.x：把 (x,y,z) -> (x,y)
    return transform(lambda x, y, z=None: (x, y), geom)

def shapely_to_ee_geometry(shp_geom):
    # 1) 先强制 2D
    shp_geom = force_2d(shp_geom)
    # 2) 修复无效几何（自相交等）
    if not shp_geom.is_valid:
        shp_geom = shp_geom.buffer(0)
    geo_json = mapping(shp_geom)
    gtype = geo_json["type"]
    coords = geo_json["coordinates"]

    if gtype == "Polygon":
        return ee.Geometry.Polygon(coords)
    elif gtype == "MultiPolygon":
        return ee.Geometry.MultiPolygon(coords)
    else:
        raise ValueError(f"Unsupported geometry type: {gtype}")
    
def cal_cloud_per(img):
    
    cloud_img = img.select('cloud')
    Cloud_count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.sum(),
      geometry = roi,
      scale = 30,
      maxPixels = 1e13
    ).get('cloud')
  
    count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.count(),
      geometry = roi,
      scale = 30,
      maxPixels =  1e13
    ).get('cloud')
    
    cloud_per = ee.Number(Cloud_count).divide(count).multiply(100)
    
    return img.set('cloud_per', cloud_per)
  
def CLIP(img):
    return img.clip(roi)

def S1_edge_remove(img):
    edgeMask = img.select('VV').gte(-30)
    return img.updateMask(edgeMask)

In [3]:
# path = r"E:\GEEdownload\UK_TF_roi\MGRS_UK_FK.shp"
# gdf = gpd.read_file(path)

# target_region = 'Region3d'
# # --- 选 Morecambe 区 ---
# region_grid = gdf[(gdf['name_1'] == target_region)]
# MGRS_list = region_grid['Name'].values
# print(f' {len(MGRS_list)} MGRS_tile were exit! ','\n', f'MGRS_list: {MGRS_list}')

# if region_grid.empty:
#     raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")

# #load edge roi
# edge_path = f'E:\GEEdownload\TK-TF_edge_collect\{target_region}_edge.shp'
edge_path = f"E:\paper5_shp\Dongsha_edge.shp"
edge_gdf = gpd.read_file(edge_path)
edge_gdf  = geemap.geopandas_to_ee(edge_gdf)       
edge_roi = edge_gdf.geometry() 


In [ ]:
path = r"E:\GEEdownload\UK_TF_roi\MGRS_UK_FK.shp"

gdf = gpd.read_file(path)
grid = region_grid = gdf[(gdf['name_1'] == 'Henber')]

merged = unary_union([force_2d(geom) for geom in grid.geometry if geom is not None])
roi_all = shapely_to_ee_geometry(merged)

In [5]:
# target_region_list = ['Thames_esturay', 'Region1a', 'Region1b', 'Region1c',
#                       'Henber', 'Region2a',
#                       'Morecambe', 'ribble', 'Severn_Esturay', 'Region4a1', 'Region4a2', 'Region4a3',
#                       'Region4a4', 'Region4b', 'Region4c', 'region4d',
#                       'Region3a1', 'Region3b1', 'Region3b2', 'Region3c',]

target_region_list = ['Beibu_bay', 'Fangcheng_gang']

year_list = [2019, 2020, 2021, 2022, 2023, 2024]
year_list = [2022]

# path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
path = r'H:/FK_ROI/FK_GD_GX.shp'
gdf = gpd.read_file(path)

for year in year_list:

    start_date, end_date = f'{year}-01-01', f'{year+1}-01-01'
    
    for target_region in target_region_list:
        
        # path = r"E:\GEEdownload\UK_TF_roi\UK_FK.shp"
        path = r'H:/FK_ROI/FK_GD_GX.shp'
        gdf = gpd.read_file(path)
        region_grid = gdf[(gdf['name'] == target_region)]
        # MGRS_TILE = region_grid['MGRS'].values[0]

        if region_grid.empty:
            raise ValueError("筛选结果为空：检查字段 'name_1' 和筛选值。")
        merged_ = unary_union([force_2d(geom) for geom in region_grid.geometry if geom is not None])
        roi = shapely_to_ee_geometry(merged_)

        # edge_path = f'E:\GEEdownload\TK-TF_edge_collect\{target_region}_edge.shp'
        # edge_path = f"E:\paper5_shp\Dongsha_edge.shp"
        # edge_gdf = gpd.read_file(edge_path)
        # edge_gdf  = geemap.geopandas_to_ee(edge_gdf)       
        edge_roi = roi#edge_gdf.geometry() 

        cloud_limit = 40
        
        S2 = F.get_s2_sr_cld_col(roi, start_date, end_date, 100)
        
        # if MGRS_TILE != 'False':
        #     print('--apply MGRS_TILE--', MGRS_TILE)
        #     S2 = S2.filter(ee.Filter.eq('MGRS_TILE', MGRS_TILE))
                    
        S2 = S2.map(F.VI_cal_S2).map(F.add_cloud_bands)\
            .map(CLIP).map(cal_cloud_per)\
            .filter(ee.Filter.gt('cloud_per', cloud_limit))
            
        L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
        L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
        L9 = F.Landsat_processing(roi, L9_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)
        L8 = F.Landsat_processing(roi, L8_collect, start_date, end_date).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40)).map(F.VI_cal_L8)

        S1_TB = F.get_s1_collection(roi, start_date, end_date).map(CLIP).map(S1_edge_remove).map(F.S1_index)
        S1_TB_mean = S1_TB.select('VV').mean()
        
        all_TF_collect = S2.merge(L8).merge(L9)
        all_TF_collect = ee.ImageCollection(all_TF_collect)
        sum_TF = all_TF_collect.map(F.TF_binary_cal).toBands().reduce(ee.Reducer.sum())
        sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
        Frequency = sum_TF.divide(sum_cloud).rename('Freq')
            
        S2_2022 = S2.map(F.normalized_TFI).map(F.TF_get)
        Norm_S2_2022 = S2_2022.map(F.Norm)
        cloud_sum = S2_2022.select('cloud').toBands().reduce(ee.Reducer.sum())

        NDWI_norm_min = S2_2022.filter(ee.Filter.gt('cloud_per', 60)).select('NDWI_norm').reduce(ee.Reducer.percentile([1])).rename('NDWI_min')
        NDWI_mean = S2_2022.select('NDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_NDWI')
        MNDWI_mean = S2_2022.select('MNDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_MNDWI')            
        composit_b8A = S2_2022.select('B8A').reduce(ee.Reducer.sum()).divide(cloud_sum).divide(10000).rename('mean_B8A')
        NDVI_mean = S2_2022.select('NDVI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_NDVI')
        S2_NDWI_min = S2_2022.select('NDWI').reduce(ee.Reducer.percentile([2])).rename('S2_NDWI_min')

        S2_NDWI_norm_min = S2_2022.filter(ee.Filter.gt('cloud_per', 60)).select('NDWI_norm').reduce(ee.Reducer.percentile([2])).rename('NDWI_min')
        # TF_mask = S2_NDWI_min.lt(-0.1).And(Frequency.gt(0.05)).And(Frequency.lt(0.95)).clip(edge_roi).unmask(0).rename('mask')
        TF_mask = (Frequency.gt(0.05)).And(Frequency.lt(0.95)).clip(edge_roi).unmask(0).rename('mask')

        feature_bands = Frequency.addBands(NDVI_mean).addBands(MNDWI_mean).addBands(composit_b8A).addBands(S1_TB_mean).addBands(S2_NDWI_min).addBands(TF_mask)
        feature_bands = feature_bands.clip(roi)
        
        filename = f'{target_region}_{year}_FB6_new9'
            
        task = ee.batch.Export.image.toDrive(
            image=feature_bands.toFloat(),
            description= filename,
            folder='paper5_FBs',
            fileNamePrefix=filename,
            region=roi,
            scale=30,
            maxPixels=1e13
        )
        task.start()
        print(f"\n✅ filename task已提交，请到 https://code.earthengine.google.com/tasks 查看进度")


✅ filename task已提交，请到 https://code.earthengine.google.com/tasks 查看进度

✅ filename task已提交，请到 https://code.earthengine.google.com/tasks 查看进度
